# Master's Thesis Run Notebook

This notebook is part of the **Master's Thesis (MSc Dissertation)**: **Fast Simulation of Neutrino Oscillations in Matter**.

**Author**  
Juan Ramon Diaz Santos <diazjuan@alumni.uv.es>

**Supervisors**  
Roberto Ruiz de Austri Bazan <rruiz@ific.uv.es>  
Michele Lucente <michele.lucente@unibo.it>

**Date**  
June 2026


# Atmosphere 1: Honda Differential Flux Generation

Generate Honda/HKKM height-differential atmospheric neutrino flux files with the same downstream contract as the MCEq production run.


## 1. Libraries

This section imports Python standard-library modules, PyTorch, repository helpers, and the TPeanuts APIs required by the run. It also locates the repository root so the notebook can be executed from Jupyter without relying on the script `__file__` variable.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import torch

from tpeanuts.external.honda import HondaTableSelection
from tpeanuts.external.honda.generator import generate_flux_for_particles_angle_grid
from tpeanuts.io.io_atmosphere import OutputConfig
from tpeanuts.util.parallel import ParallelConfig
from tpeanuts.atmosphere.geometry import detector_alpha_to_surface_theta
from tpeanuts.io.io_atmosphere import load_directory
from tpeanuts.util.torch_util import _default_device, resolve_device
from tpeanuts.util.type import _as_tensor

from tpeanuts.util.notebooks import find_repo_root
HERE = Path.cwd().resolve()
PACKAGE_DIR = find_repo_root(HERE, folder="analysis")
print(f"PACKAGE_DIR = {PACKAGE_DIR}")


PACKAGE_DIR = G:\Mi unidad\03.Codigo\034.TFM.UV\Tpeanuts


## 2. Paths and Configuration

This section centralizes every filesystem path and all run parameters before any computation is started. Keeping these values together makes the run reproducible and avoids hidden output locations.


### 2.1. Paths

All outputs are rooted at `DEFAULT_OUTPUT_ROOT` unless the environment variable `TPEANUTS_OUTPUT_ROOT` is set. Derived folders are created from that single root, with atmospheric production outputs written under `OUTPUT_DATA_ROOT / "atmosphere"`.


In [2]:
DEFAULT_HONDA_DATA_DIR = Path(r"G:\Mi unidad\03.Codigo\034.TFM.UV\External\Honda")
HONDA_DATA_DIR = Path(os.environ.get("HONDA_DATA_DIR", DEFAULT_HONDA_DATA_DIR))

DEFAULT_OUTPUT_ROOT = Path(r"V:\output")
OUTPUT_ROOT = Path(os.environ.get("TPEANUTS_OUTPUT_ROOT", DEFAULT_OUTPUT_ROOT))

OUTPUT_DATA_ROOT = Path(OUTPUT_ROOT / "data")
OUTPUT_ANALYSIS_ROOT = Path(OUTPUT_ROOT / "analysis")
OUTPUT_BENCHMARK_ROOT = Path(OUTPUT_ROOT / "benchmark")
OUTPUT_TEST_ROOT = Path(OUTPUT_ROOT / "test")

OUTPUT_DATA_ATMOSPHERE = Path(OUTPUT_DATA_ROOT / "atmosphere")
OUTPUT_DATA_SOLAR = Path(OUTPUT_DATA_ROOT / "solar")
OUTPUT_DATA_EXTERNAL = Path(OUTPUT_DATA_ROOT / "external")

OUTPUT_ANALYSIS_ATMOSPHERE = Path(OUTPUT_ANALYSIS_ROOT / "atmosphere")
OUTPUT_ANALYSIS_SOLAR = Path(OUTPUT_ANALYSIS_ROOT / "solar")
OUTPUT_ANALYSIS_EXTERNAL = Path(OUTPUT_ANALYSIS_ROOT / "external")

OUTPUT_DATA_MCEQ = Path(OUTPUT_DATA_EXTERNAL / "mceq")
OUTPUT_DATA_HONDA = Path(OUTPUT_DATA_EXTERNAL / "honda")

OUTPUT_ATMOSPHERE_ROOT = Path(OUTPUT_DATA_ATMOSPHERE)
OUTPUT_HONDA_ROOT = Path(OUTPUT_ATMOSPHERE_ROOT / "honda")
OUTPUT_DIR = Path(OUTPUT_HONDA_ROOT / "test")

OUTPUT_FILENAME = "diff_flux.pt"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Output Atmospheric Honda Differential Flux Data:', OUTPUT_DIR, ' --> Path Exists = ', os.path.exists( OUTPUT_DIR))

Output Atmospheric Honda Differential Flux Data: V:\output\data\atmosphere\honda\test  --> Path Exists =  True


## 2.2. Parameters and System Configuration

This subsection defines the system configuration used by the run. The parameters control the physical model, angular and numerical grids, output precision, runtime device, batching strategy, and overwrite/skip behavior. Adjust these values before executing the run cells when producing a different dataset. The expected result is a consistent set of validated configuration objects and reproducible output paths. Possible problems include unavailable external data, missing MCEq models, excessive memory use from large chunks, or accidental reuse of existing files when `SKIP_EXISTING` is enabled.


### Data products

These parameters define the data products written by this run. Review them before executing production cells, especially when changing datasets, precision, or runtime size.


In [3]:
SAVE_DTYPE = torch.float32

OVERWRITE = False

HONDA_SELECTION = HondaTableSelection(
    site_code="frj",
    season_code="ally",
    solar="solmin",
    mountain=False,
    angular_mode="azimuth-averaged",
    azimuth_averaged_height=True,
)

PARTICLES = {
    "nue": "nue",
    "antinue": "antinue",
    "numu": "numu",
    "antinumu": "antinumu",
    "nutau": "nutau",
    "antinutau": "antinutau",
}


### Angle convention

These parameters define the angle convention used by this run. Review them before executing production cells, especially when changing datasets, precision, or runtime size.


In [4]:
USE_HONDA_COSZ_GRID = True

USE_DETECTOR_ALPHA_GRID = False

HONDA_COSZ_CENTERS = torch.linspace(0.95, 0.05, 10)

ALPHA_MIN = 0

ALPHA_MAX = 180

ALPHA_N = 11

ALPHA_DETECTOR_GRID_DEG = torch.linspace(ALPHA_MIN, ALPHA_MAX, ALPHA_N)

THETA_MIN = 0

THETA_MAX = 89.5

THETA_N = 37

THETA_SURFACE_GRID_DEG = torch.linspace(THETA_MIN, THETA_MAX, THETA_N)

DETECTOR_DEPTH_M = 1000.0

SURFACE_THETA_MAX_DEG = 89.999


### Numerical grids

These parameters define the numerical grids used by this run. Review them before executing production cells, especially when changing datasets, precision, or runtime size.


In [5]:
ENERGY_GRID_GEV = None

H_GRID_MIN = 0.0

H_GRID_MAX = 120.0

H_GRID_N = 501

H_GRID_KM = torch.linspace(H_GRID_MIN, H_GRID_MAX, H_GRID_N)


### Runtime and batching

These parameters control runtime behavior and batching for this run. Review them before executing production cells, especially when changing datasets, precision, or runtime size.


In [6]:
GENERATION_DEVICE = "cpu"

COMPUTE_DTYPE = torch.float64

PARALLEL = False

N_JOBS = 4

PARALLEL_BACKEND = "loky"

STACK_AFTER_GENERATION = True

STACK_DEVICE = _default_device

STACK_DTYPE = torch.float64

STACK_GROUP_BY = "particle"

SAVE = True

SKIP_EXISTING = True

DEBUG = True


## 3. Helper functions

The following cells define the helper functions used by the run. Each function is kept in its own code cell so it can be inspected, edited, and rerun independently during interactive validation.


### Function: `prepared_angle_grids`

This helper prepares or transforms part of the run state used by `main()`. The expected result is that it returns validated tensors, configuration objects, paths, or partial results without writing unrelated files.


In [7]:
def prepared_angle_grids():
    if USE_HONDA_COSZ_GRID:
        cosz_grid = _as_tensor(HONDA_COSZ_CENTERS, device="cpu", dtype=torch.float64).reshape(-1)
        valid = (cosz_grid > 0.0) & (cosz_grid <= 1.0)
        if not torch.all(valid):
            dropped = int((~valid).sum().item())
            print(f"Dropping {dropped} Honda cosZ centres outside (0, 1].")
            cosz_grid = cosz_grid[valid]

        theta_grid = torch.rad2deg(torch.acos(torch.clamp(cosz_grid, -1.0, 1.0)))
        valid_theta = (theta_grid >= 0.0) & (theta_grid < SURFACE_THETA_MAX_DEG)
        theta_grid = theta_grid[valid_theta]
        theta_grid = theta_grid[torch.argsort(theta_grid)]
        return None, theta_grid, "honda-cosz theta"

    if USE_DETECTOR_ALPHA_GRID:
        alpha_grid = _as_tensor(ALPHA_DETECTOR_GRID_DEG, device="cpu", dtype=torch.float64).reshape(-1)
        theta_grid = detector_alpha_to_surface_theta(
            alpha_grid,
            detector_depth_m=DETECTOR_DEPTH_M,
            device="cpu",
            dtype=torch.float64,
        ).reshape(-1)

        valid = (theta_grid >= 0.0) & (theta_grid < SURFACE_THETA_MAX_DEG)
        if not torch.all(valid):
            dropped = int((~valid).sum().item())
            print(
                f"Dropping {dropped} detector-alpha values whose surface "
                f"theta is outside [0, {SURFACE_THETA_MAX_DEG}) deg."
            )
            alpha_grid = alpha_grid[valid]

        alpha_grid = alpha_grid[torch.argsort(alpha_grid)]
        return alpha_grid, None, "alpha"

    theta_grid = _as_tensor(THETA_SURFACE_GRID_DEG, device="cpu", dtype=torch.float64).reshape(-1)
    valid = (theta_grid >= 0.0) & (theta_grid < SURFACE_THETA_MAX_DEG)
    theta_grid = theta_grid[valid]
    theta_grid = theta_grid[torch.argsort(theta_grid)]
    return None, theta_grid, "theta"


### Function: `build_configs`

This helper prepares or transforms part of the run state used by `main()`. The expected result is that it returns validated tensors, configuration objects, paths, or partial results without writing unrelated files.


In [8]:
def build_configs():
    output_config = OutputConfig(
        output_dir=OUTPUT_DIR,
        filename=OUTPUT_FILENAME,
        dtype=SAVE_DTYPE,
        compressed=True,
        overwrite=OVERWRITE,
        save_intermediate=False,
    )
    parallel_config = ParallelConfig(
        parallel=PARALLEL,
        n_jobs=N_JOBS,
        backend=PARALLEL_BACKEND,
    )

    output_config.validate()
    parallel_config.validate()
    return output_config, parallel_config


## 3. Main run function

`main()` coordinates the configured workflow: it validates parameters, loads or generates input data, dispatches the numerical computation, writes output files, and returns a compact summary object.


In [9]:
def main():
    output_config, parallel_config = build_configs()
    alpha_grid, theta_grid, angle_mode = prepared_angle_grids()

    generation_device = resolve_device(GENERATION_DEVICE)
    stack_device = resolve_device(STACK_DEVICE)

    print("\nHonda flux generation")
    print(f"Honda data dir   : {HONDA_DATA_DIR}")
    print(f"Output directory : {OUTPUT_DIR}")
    print(f"Selection        : {HONDA_SELECTION}")
    print(f"Particles        : {PARTICLES}")
    print(f"Detector depth m : {DETECTOR_DEPTH_M}")
    print(f"Angle mode       : {angle_mode}")
    print(f"Generation device: {generation_device}")
    print(f"Compute dtype    : {COMPUTE_DTYPE}")
    print(f"Save dtype       : {SAVE_DTYPE}")

    results = generate_flux_for_particles_angle_grid(
        particles=PARTICLES,
        alpha_grid_deg=alpha_grid,
        theta_grid_deg=theta_grid,
        detector_depth_m=DETECTOR_DEPTH_M,
        honda_data_dir=HONDA_DATA_DIR,
        selection=HONDA_SELECTION,
        energy_grid_GeV=ENERGY_GRID_GEV,
        h_grid_km=H_GRID_KM,
        output_config=output_config,
        parallel_config=parallel_config,
        save=SAVE,
        skip_existing=SKIP_EXISTING,
        device=generation_device,
        dtype=COMPUTE_DTYPE,
        debug=DEBUG,
    )

    n_files = sum(len(item["results"]) for item in results.values())
    print(f"\nFinished. Generated/visited {n_files} particle-angle results.")

    stacked = None
    if STACK_AFTER_GENERATION:
        print(
            "\nLoading generated files as batched tensors "
            f"(device={stack_device}, dtype={STACK_DTYPE}, group_by={STACK_GROUP_BY})"
        )

        stacked = load_directory(
            OUTPUT_DIR,
            map_location="cpu",
            dtype=STACK_DTYPE,
            device=stack_device,
            group_by=STACK_GROUP_BY,
            verbose=DEBUG,
        )

        for particle, data in stacked.items():
            print(
                f"Stacked {particle:12s}: "
                f"phi_E_theta_h={tuple(data['phi_E_theta_h'].shape)}, "
                f"device={data['phi_E_theta_h'].device}"
            )

    return {
        "generation_results": results,
        "stacked": stacked,
    }


## 4. Execute run

The final cell launches the configured production run and stores the returned object in `RUN_RESULT`. Generation notebooks return the generation metadata and optional stacked tensors; propagation notebooks return the visited or saved detector-flux paths. Keep `DEBUG=True` while validating paths and reduce it for long production runs.


In [10]:
RUN_RESULT = main()
RUN_RESULT



Honda flux generation
Honda data dir   : G:\Mi unidad\03.Codigo\034.TFM.UV\External\Honda
Output directory : V:\output\data\atmosphere\honda\test
Selection        : HondaTableSelection(site_code='frj', season_code='ally', solar='solmin', mountain=False, angular_mode='azimuth-averaged', azimuth_averaged_height=True)
Particles        : {'nue': 'nue', 'antinue': 'antinue', 'numu': 'numu', 'antinumu': 'antinumu', 'nutau': 'nutau', 'antinutau': 'antinutau'}
Detector depth m : 1000.0
Angle mode       : honda-cosz theta
Generation device: cpu
Compute dtype    : torch.float64
Save dtype       : torch.float32
Generating 60 Honda flux jobs (6 particles x 10 angles). parallel=False


Honda flux generation:   5%|███                                                         | 3/60 [00:00<00:02, 25.21it/s]

Saved Honda nue: V:\output\data\atmosphere\honda\test\diff_flux_nue_theta_18p195deg.pt. Time: 0.030 s.
Saved Honda nue: V:\output\data\atmosphere\honda\test\diff_flux_nue_theta_31p788deg.pt. Time: 0.030 s.
Saved Honda nue: V:\output\data\atmosphere\honda\test\diff_flux_nue_theta_41p410deg.pt. Time: 0.034 s.
Saved Honda nue: V:\output\data\atmosphere\honda\test\diff_flux_nue_theta_49p458deg.pt. Time: 0.038 s.
Saved Honda nue: V:\output\data\atmosphere\honda\test\diff_flux_nue_theta_56p633deg.pt. Time: 0.041 s.


Honda flux generation:  15%|█████████                                                   | 9/60 [00:00<00:02, 24.22it/s]

Saved Honda nue: V:\output\data\atmosphere\honda\test\diff_flux_nue_theta_63p256deg.pt. Time: 0.040 s.
Saved Honda nue: V:\output\data\atmosphere\honda\test\diff_flux_nue_theta_69p513deg.pt. Time: 0.033 s.
Saved Honda nue: V:\output\data\atmosphere\honda\test\diff_flux_nue_theta_75p522deg.pt. Time: 0.034 s.
Saved Honda nue: V:\output\data\atmosphere\honda\test\diff_flux_nue_theta_81p373deg.pt. Time: 0.031 s.
Saved Honda nue: V:\output\data\atmosphere\honda\test\diff_flux_nue_theta_87p134deg.pt. Time: 0.032 s.
Saved Honda antinue: V:\output\data\atmosphere\honda\test\diff_flux_antinue_theta_18p195deg.pt. Time: 0.029 s.


...

Honda flux generation:  95%|████████████████████████████████████████████████████████   | 57/60 [00:02<00:00, 25.03it/s]

Saved Honda antinutau: V:\output\data\atmosphere\honda\test\diff_flux_antinutau_theta_49p458deg.pt. Time: 0.037 s.
Saved Honda antinutau: V:\output\data\atmosphere\honda\test\diff_flux_antinutau_theta_56p633deg.pt. Time: 0.030 s.
Saved Honda antinutau: V:\output\data\atmosphere\honda\test\diff_flux_antinutau_theta_63p256deg.pt. Time: 0.029 s.
Saved Honda antinutau: V:\output\data\atmosphere\honda\test\diff_flux_antinutau_theta_69p513deg.pt. Time: 0.033 s.
Saved Honda antinutau: V:\output\data\atmosphere\honda\test\diff_flux_antinutau_theta_75p522deg.pt. Time: 0.030 s.
Saved Honda antinutau: V:\output\data\atmosphere\honda\test\diff_flux_antinutau_theta_81p373deg.pt. Time: 0.032 s.
Saved Honda antinutau: V:\output\data\atmosphere\honda\test\diff_flux_antinutau_theta_87p134deg.pt. Time: 0.029 s.


Honda flux generation: 100%|███████████████████████████████████████████████████████████| 60/60 [00:02<00:00, 24.71it/s]



Finished. Generated/visited 60 particle-angle results.

Loading generated files as batched tensors (device=cuda:0, dtype=torch.float64, group_by=particle)
Loaded antinue      | n_theta =  10 | phi(E,theta,h) shape = (10, 101, 501)
Loaded antinumu     | n_theta =  10 | phi(E,theta,h) shape = (10, 101, 501)
Loaded antinutau    | n_theta =  10 | phi(E,theta,h) shape = (10, 101, 501)
Loaded nue          | n_theta =  10 | phi(E,theta,h) shape = (10, 101, 501)
Loaded numu         | n_theta =  10 | phi(E,theta,h) shape = (10, 101, 501)
Loaded nutau        | n_theta =  10 | phi(E,theta,h) shape = (10, 101, 501)
Stacked antinue     : phi_E_theta_h=(10, 101, 501), device=cuda:0
Stacked antinumu    : phi_E_theta_h=(10, 101, 501), device=cuda:0
Stacked antinutau   : phi_E_theta_h=(10, 101, 501), device=cuda:0
Stacked nue         : phi_E_theta_h=(10, 101, 501), device=cuda:0
Stacked numu        : phi_E_theta_h=(10, 101, 501), device=cuda:0
Stacked nutau       : phi_E_theta_h=(10, 101, 501), device

{'generation_results': {'nue': {'particle': 'nue',
   'flavour_name': 'nue',
   'angle_mode': 'theta',
   'input_angle_grid_deg': tensor([18.1949, 31.7883, 41.4096, 49.4584, 56.6330, 63.2563, 69.5127, 75.5225,
           81.3731, 87.1340], dtype=torch.float64),
   'results': {18.194874526177813: {'particle': 'nue',
     'flavour_name': 'nue',
     'theta_deg': tensor(18.1949, dtype=torch.float64),
     'E_grid_GeV': tensor([1.0000e-01, 1.1220e-01, 1.2589e-01, 1.4125e-01, 1.5849e-01, 1.7783e-01,
             1.9953e-01, 2.2387e-01, 2.5119e-01, 2.8184e-01, 3.1623e-01, 3.5481e-01,
             3.9811e-01, 4.4668e-01, 5.0119e-01, 5.6234e-01, 6.3096e-01, 7.0795e-01,
             7.9433e-01, 8.9125e-01, 1.0000e+00, 1.1220e+00, 1.2589e+00, 1.4125e+00,
             1.5849e+00, 1.7783e+00, 1.9953e+00, 2.2387e+00, 2.5119e+00, 2.8184e+00,
             3.1623e+00, 3.5481e+00, 3.9811e+00, 4.4668e+00, 5.0119e+00, 5.6234e+00,
             6.3096e+00, 7.0795e+00, 7.9433e+00, 8.9125e+00, 1.0000e+01, 1.